# Data Augmentation para Landmarks de Mãos/Braços (LIBRAS)

Este notebook aplica estratégias de data augmentation em arquivos de landmarks gerados pelo pipeline do notebook de coleta (`create_landmarks.ipynb`).

- Compatível com o padrão de arquivos `.npy` (dict com 'hands' e opcionalmente 'arms', ou apenas array de mãos).
- Estratégias aplicadas:
  - Ruído Gaussiano controlado
  - Escala leve
  - Rotação suave (eixo Z)
  - Time warping (para sequências)

In [ ]:
import os
import numpy as np
import glob
import shutil

# Caminhos
LANDMARKS_DIR = '../dataset/processed/landmarks'
AUGMENTED_DIR = '../dataset/processed/landmarks_augmented'

os.makedirs(AUGMENTED_DIR, exist_ok=True)

# Funções utilitárias para augmentations

def add_gaussian_noise(landmarks, std=0.005):
    noise = np.random.normal(0, std, landmarks.shape)
    return landmarks + noise

def scale_landmarks(landmarks, scale_range=(0.95, 1.05)):
    center = landmarks.mean(axis=0, keepdims=True)
    scale = np.random.uniform(*scale_range)
    return (landmarks - center) * scale + center

def rotate_landmarks_z(landmarks, angle_range=(-10, 10)):
    theta = np.radians(np.random.uniform(*angle_range))
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    center = landmarks[:, :2].mean(axis=0, keepdims=True)
    xy = landmarks[:, :2] - center
    xy_rot = xy @ R.T + center
    landmarks_aug = landmarks.copy()
    landmarks_aug[:, :2] = xy_rot
    return landmarks_aug

def time_warp_sequence(seq, max_warp=0.1):
    # seq: (frames, ...)
    n = seq.shape[0]
    if n < 3:
        return seq  # não warpa sequências muito curtas
    # Compressão/expansão leve
    factor = np.random.uniform(1 - max_warp, 1 + max_warp)
    idxs = np.linspace(0, n-1, int(n * factor)).clip(0, n-1)
    idxs = np.round(idxs).astype(int)
    seq_warp = seq[idxs]
    # Frame dropping aleatório
    if len(seq_warp) > 3 and np.random.rand() < 0.5:
        drop_idx = np.random.randint(1, len(seq_warp)-1)
        seq_warp = np.delete(seq_warp, drop_idx, axis=0)
    return seq_warp

def augment_landmarks(landmarks):
    # Aplica todas as estratégias em sequência
    lm = landmarks.copy()
    lm = add_gaussian_noise(lm)
    lm = scale_landmarks(lm)
    lm = rotate_landmarks_z(lm)
    return lm

def augment_file(file_path, save_dir, n_aug=3):
    arr = np.load(file_path, allow_pickle=True)
    base = os.path.splitext(os.path.basename(file_path))[0]
    # Detecta padrão do arquivo
    if isinstance(arr, np.ndarray) and arr.dtype == object:
        arr = arr.item()
    if isinstance(arr, dict) and 'hands' in arr:
        hands = arr['hands']
        arms = arr['arms'] if 'arms' in arr else None
        # Sequência temporal
        if hands.ndim == 4:
            for i in range(n_aug):
                hands_aug = np.stack([augment_landmarks(frame) for frame in hands])
                # Time warping
                if np.random.rand() < 0.5:
                    hands_aug = time_warp_sequence(hands_aug)
                    if arms is not None:
                        arms_aug = time_warp_sequence(arms)
                    else:
                        arms_aug = None
                else:
                    arms_aug = arms
                out = {'hands': hands_aug}
                if arms_aug is not None:
                    out['arms'] = arms_aug
                np.save(os.path.join(save_dir, f'{base}_aug{i+1}.npy'), out)
        # Imagem
        elif hands.ndim == 3:
            for i in range(n_aug):
                hands_aug = augment_landmarks(hands)
                out = {'hands': hands_aug}
                if arms is not None:
                    out['arms'] = arms
                np.save(os.path.join(save_dir, f'{base}_aug{i+1}.npy'), out)
    else:
        # Compatibilidade: (seq_len, 21, 3) ou (2, 21, 3)
        if arr.ndim == 3 and arr.shape[0] == 2 and arr.shape[1] == 21:
            arr = arr[np.newaxis, ...]
        # Sequência
        if arr.ndim == 4:
            for i in range(n_aug):
                arr_aug = np.stack([augment_landmarks(frame) for frame in arr])
                arr_aug = time_warp_sequence(arr_aug) if np.random.rand() < 0.5 else arr_aug
                np.save(os.path.join(save_dir, f'{base}_aug{i+1}.npy'), arr_aug)
        # Imagem
        elif arr.ndim == 3:
            for i in range(n_aug):
                arr_aug = augment_landmarks(arr)
                np.save(os.path.join(save_dir, f'{base}_aug{i+1}.npy'), arr_aug)

# Processa todos os arquivos de landmarks
files = glob.glob(os.path.join(LANDMARKS_DIR, '*.npy'))
print(f'Encontrados {len(files)} arquivos para augmentar.')
for f in files:
    augment_file(f, AUGMENTED_DIR, n_aug=3)
print('Data augmentation finalizado. Arquivos salvos em', AUGMENTED_DIR)

## Observações
- Os arquivos aumentados são salvos em `../dataset/processed/landmarks_augmented` com sufixo `_aug1`, `_aug2`, ...
- O padrão de estrutura dos arquivos é mantido igual ao pipeline de coleta.
- Recomenda-se atualizar o CSV de metadados após o augmentation, caso queira treinar com os dados aumentados.